# 55. PDF 온도 변화 확인 후보7/6 감사
54번 실제 예측의 TP/FP/FN/TN 재집계, 원본 정답, 기존 판정유지, 학습·평가 구간 분리, 과거평가 미래 제외, 추가2개가 과거평가 대상인지 확인한다. 두 가스는 가장 이른 가스 관측구간에 속해 과거2클래스학습 평가에서 제외된다. 외부최고 조합을 독립 검증으로 해석하지 않는다.


In [ ]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
raw=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=raw[(raw['product']=='RG3')&((raw.PassOrFail==0)|raw.Reason.str.contains('가스',na=False))].reset_index(drop=True)
p=pd.read_csv(tables/'rg3_pdf_confirm_gates_predictions.csv',parse_dates=['TimeStamp'])
c=pd.read_csv(tables/'rg3_pdf_confirm_gates_inner_candidates.csv')
primary=p[p.model.eq('thermal_change_low_0.05')].copy()
verify=[]
for scheme,g in primary.groupby('scheme'):
 assert not g.TimeStamp.duplicated().any()
 assert np.array_equal(d.set_index('TimeStamp').loc[g.TimeStamp,'PassOrFail'].to_numpy(),g.actual.to_numpy())
 assert ((g.anchor_prediction==0)|(g.prediction==1)).all()
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 verify.append(dict(scheme=scheme,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn),predicted_defect=int(tp+fp)))
whole=primary[primary.scheme=='leave_one_block_out']
assert len(whole)==583
assert (whole.actual.sum(),whole.prediction.sum())==(17,13)
new=whole[(whole.actual==1)&(whole.prediction==1)&(whole.anchor_prediction==0)]
assert len(new)==2
cut=c[c.gate.eq('thermal_change_low_0.05')][['scheme','block','gate_thresholds']]
whole=whole.merge(cut,on=['scheme','block'],validate='many_to_one')
whole['thermal_confirm_threshold']=whole.gate_thresholds.map(lambda v:json.loads(v)[0])
whole['past_evaluation_eligible']=whole.TimeStamp.isin(primary.loc[primary.scheme=='past_blocks_only','TimeStamp'])
whole[whole.actual.eq(1)].to_csv(tables/'rg3_pdf_confirm_all17.csv',index=False,encoding='utf-8-sig')
whole.to_csv(tables/'rg3_pdf_confirm_all583.csv',index=False,encoding='utf-8-sig')
v=pd.DataFrame(verify);v.to_csv(tables/'rg3_pdf_confirm_verification.csv',index=False,encoding='utf-8-sig')
fold=pd.read_csv(tables/'rg3_pdf_confirm_gates_fold_audit.csv')
assert fold.timestamp_overlap.eq(0).all()
# Derive outer reference groups again, not just trust stored overlap counts.
blocks=d.comparison_block.drop_duplicates().tolist()
for row in fold.itertuples():
 q=d[d.comparison_block.eq(row.block)]
 j=blocks.index(row.block)
 ref=d[~d.comparison_block.eq(row.block)] if row.scheme=='leave_one_block_out' else d[d.comparison_block.isin(blocks[:j])]
 assert len(ref)==row.train and len(q)==row.test
 assert set(q.TimeStamp).isdisjoint(ref.TimeStamp)
 assert set(q.comparison_block).isdisjoint(ref.comparison_block)
 if row.scheme=='past_blocks_only':assert ref.TimeStamp.max()<q.TimeStamp.min()
print(v.to_string(index=False))
print('New gases and time-forward coverage')
print(whole[(whole.actual==1)&(whole.prediction==1)&(whole.anchor_prediction==0)][['TimeStamp','score','score_threshold','gate_value_thermal','thermal_confirm_threshold','past_evaluation_eligible']].to_string(index=False))
print('Gas counts by actual labeled-observation block')
print(d.groupby('comparison_block').PassOrFail.agg(['count','sum']).to_string())
